# 08 - Crypto High-Frequency VPIN & Order Flow Toxicity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huuan26/polars-anta/blob/master/examples/08_crypto_vpin_order_flow_toxicity.ipynb)

Crypto books reprice in milliseconds, but the interesting question is not *how fast* prices move - it is *who* is moving them. This notebook exercises the two microstructure primitives that institutional desks watch on high-frequency crypto tape:

- **VPIN** (Volume-Synchronized Probability of Informed Trading, Easley / Lopez de Prado / O'Hara 2012) - how much of the traded volume is likely informed flow, sampled on **volume bars** instead of clock time.
- **OFI** (Order Flow Imbalance, Cont / Kukanov / Stoikov 2014) - net queue pressure at the best bid and ask, and whether it leads forward returns.

Everything runs on a synthetic tape generated locally: no exchange keys, no downloads. Because we generate the informed flow ourselves, we can check that the estimators recover it - something real tape never lets us do.


In [ ]:
# Install polars-anta (uncomment if running in Colab)
!pip install -q polars polars-anta matplotlib

In [ ]:
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

import polars_anta as pa

print(f"Polars Version      : {pl.__version__}")
print(f"polars-anta Version : {pa.__version__}")

## 1. Simulate a Crypto Tape with Informed-Trading Bursts

Six hours of one-second prints on a BTC perpetual, built from two latent processes:

1. **Order-flow pressure** - an AR(1) process `flow` that drives both quote sizes and the next-second return. This is the structure OFI is supposed to recover.
2. **Informed bursts** - four one-sided windows where a larger fraction of prints is informed and pushes the mid in one direction. This is the structure VPIN is supposed to detect.

Uninformed prints are balanced noise. The only link between flow and future returns is the one we put there, so any correlation below is evidence about the estimator, not about a cherry-picked price path.


In [ ]:
rng = np.random.default_rng(59)
n_seconds = 21_600  # six hours of one-second prints
start = datetime(2026, 3, 2, 9, 0)

# 1) Persistent order-flow pressure: AR(1), the state OFI is meant to see.
flow = np.zeros(n_seconds)
for i in range(1, n_seconds):
    flow[i] = 0.95 * flow[i - 1] + rng.normal(0.0, 1.0)
flow /= flow.std()

# 2) Four one-sided informed bursts (offset, length, peak intensity, direction).
intensity = np.full(n_seconds, 0.05)
burst_mask = np.zeros(n_seconds, dtype=bool)
burst_side = np.zeros(n_seconds)
for offset, length, peak, sign in [
    (3_600, 900, 0.55, 1.0),
    (8_200, 700, 0.70, -1.0),
    (13_000, 1_100, 0.45, 1.0),
    (17_600, 800, 0.60, -1.0),
]:
    intensity[offset : offset + length] += peak * np.exp(-np.linspace(0.0, 3.0, length))
    burst_mask[offset : offset + length] = True
    burst_side[offset : offset + length] = sign

informed = rng.random(n_seconds) < np.clip(intensity, 0.0, 0.95)

# 3) Returns: flow pressure + one-sided informed impact + noise.
returns = (
    1.2e-5 * flow
    + burst_side * informed * np.abs(rng.normal(1.5e-4, 4.5e-5, n_seconds))
    + rng.normal(0.0, 2.5e-5, n_seconds)
)
price = 30_000.0 * np.exp(np.cumsum(returns))

# Aggressor side and trade size: informed prints are one-sided and larger.
side = np.where(
    informed, np.where(burst_side > 0, "buy", "sell"), np.where(flow > 0, "buy", "sell")
)
size = (
    np.where(informed, rng.lognormal(0.7, 0.6, n_seconds), rng.lognormal(-0.5, 0.5, n_seconds))
    * 0.01
)

trades = pl.DataFrame(
    {
        "timestamp": np.arange(n_seconds),  # seconds since the 09:00 session open
        "close": price,
        "volume": size,
        "side": side,
    }
)
print(f"{trades.height:,} one-second prints | {trades['volume'].sum():,.1f} BTC traded")
print(f"informed share: {informed.mean():.1%} | burst seconds: {burst_mask.mean():.1%}")
trades.head(3)

## 2. Information-Driven Resampling: Volume Bars

Clock bars hide the feature we care about: a quiet minute and a liquidation cascade both get one bar. `df.anta.resample_volume_bars(...)` emits a new bar every time cumulative traded volume crosses the threshold, so each bar carries roughly the same amount of information.

We target ~220 bars, which leaves enough history for a 25-bar VPIN window.


In [ ]:
bar_threshold = float(round(trades["volume"].sum() / 220, 3))

volume_bars = trades.anta.resample_volume_bars(
    threshold=bar_threshold,
    volume_col="volume",
    close_col="close",
    time_col="timestamp",
)

print(f"trades     : {trades.height:,}")
print(f"volume bars: {volume_bars.height:,}  (~{bar_threshold:.3f} BTC per bar)")
volume_bars.select("timestamp", "open", "high", "low", "close", "volume", "vwap", "n_obs").head(3)

## 3. VPIN: Probability of Informed Trading

VPIN classifies each bar's volume into buy- and sell-initiated flow with a Gaussian CDF of standardized price changes (Bulk Volume Classification), then reports the rolling ratio of absolute imbalance to total volume:

$$\mathrm{VPIN}_t = \frac{\sum_{i=t-n+1}^{t} |V_i^B - V_i^S|}{\sum_{i=t-n+1}^{t} V_i} \in [0, 1]$$

The first `period - 1` bars are warm-up and stay `null`. To keep regime flagging strictly causal (zero look-ahead), we compare each bar's VPIN against its trailing 60-bar 80th percentile (`vpin_threshold`): bars breaching this threshold mark a **toxic regime** (elevated informed participation, wider adverse selection, tighter risk limits).

Because the burst windows are known, we can score the detector directly: mean VPIN inside a burst should sit well above the quiet tape.


In [ ]:
VPIN_PERIOD = 25
THRESHOLD_WINDOW = 60

bars = (
    volume_bars.with_columns(pl.col("close").anta.vpin("volume", period=VPIN_PERIOD).alias("vpin"))
    .with_columns(
        pl.col("vpin")
        .rolling_quantile(0.8, window_size=THRESHOLD_WINDOW, min_samples=15)
        .alias("vpin_threshold")
    )
    .with_columns((pl.col("vpin") >= pl.col("vpin_threshold")).fill_null(False).alias("toxic"))
)

vpin_values = bars["vpin"].to_numpy()
vpin_threshold_values = bars["vpin_threshold"].to_numpy()
in_burst = burst_mask[bars["timestamp"].to_numpy().astype(int)]

print(f"VPIN period         : {VPIN_PERIOD} bars")
print(f"warm-up bars        : {bars['vpin'].null_count()}")
print(f"toxic bars flagged  : {int(bars['toxic'].sum())} (causal trailing 80th pct)")
print(f"mean VPIN | burst   : {np.nanmean(vpin_values[in_burst]):.3f}")
print(f"mean VPIN | quiet   : {np.nanmean(vpin_values[~in_burst]):.3f}")
bars.select("timestamp", "close", "volume", "vpin", "vpin_threshold", "toxic").tail(3)

## 4. Order Flow Imbalance and Forward Returns

VPIN says *how much* of the flow is informed; OFI says *which side* is pushing. At the top of book:

$$e_t = \mathbf{1}_{\{P_t^B \ge P_{t-1}^B\}} q_t^B - \mathbf{1}_{\{P_t^B \le P_{t-1}^B\}} q_{t-1}^B - \mathbf{1}_{\{P_t^A \le P_{t-1}^A\}} q_t^A + \mathbf{1}_{\{P_t^A \ge P_{t-1}^A\}} q_{t-1}^A$$

The tape only gives us trade prices, so we reconstruct a symmetric L1 book (±1.5 USD around the print) whose queue sizes lean with the latent `flow` state - the same state that drives the next-second return. Then we normalize OFI by total displayed size and line it up against the next 60-second return.


In [ ]:
HALF_SPREAD = 1.5  # USD - a tight BTC perpetual book
flow_clipped = np.clip(flow, -2.5, 2.5)
base_size = rng.lognormal(1.2, 0.5, n_seconds)

quotes = trades.with_columns(
    (pl.col("close") - HALF_SPREAD).alias("bid_price"),
    (pl.col("close") + HALF_SPREAD).alias("ask_price"),
    pl.Series("bid_size", base_size * (1.0 + 0.5 * flow_clipped)),
    pl.Series("ask_size", base_size * (1.0 - 0.5 * flow_clipped)),
)

ofi = (
    quotes.with_columns(
        pl.col("bid_price").anta.ofi("bid_size", "ask_price", "ask_size").alias("ofi")
    )
    .with_columns((pl.col("ofi") / (pl.col("bid_size") + pl.col("ask_size"))).alias("ofi_norm"))
    .with_columns((pl.col("close").shift(-60) / pl.col("close") - 1.0).alias("fwd_ret_60s"))
)

corr_60s = ofi.select(pl.corr("ofi_norm", "fwd_ret_60s")).item()
print(f"corr(normalized OFI, next-60s return) = {corr_60s:+.3f}")
print(f"usable prints                          = {ofi['ofi_norm'].is_not_null().sum():,}")

In [ ]:
valid = ofi.drop_nulls(["ofi_norm", "fwd_ret_60s"])

quintiles = (
    valid.with_columns(
        ((pl.col("ofi_norm").rank("ordinal") - 1) * 5 // valid.height)
        .cast(pl.Int8)
        .alias("quintile")
    )
    .group_by("quintile")
    .agg(
        pl.col("ofi_norm").mean().round(3).alias("mean_ofi"),
        pl.col("fwd_ret_60s").mean().mul(1e4).round(3).alias("mean_fwd_bps"),
        pl.len().alias("n_obs"),
    )
    .sort("quintile")
)
quintiles

## 5. Visualization: Toxicity Regimes and Flow Pressure

Top panel: the price path with every high-VPIN bar shaded. Bottom panel: VPIN against its 80th-percentile threshold. The shaded blocks should sit inside the four informed bursts.


In [ ]:
fig, (ax_price, ax_vpin) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.2, 1.0]}
)

bar_stamps = np.array([start + timedelta(seconds=int(s)) for s in bars["timestamp"]])
bar_seconds = int(np.median(np.diff(bars["timestamp"].to_numpy()).astype(float)))

for t in bar_stamps[bars["toxic"].to_numpy()]:
    ax_price.axvspan(t, t + timedelta(seconds=bar_seconds), color="crimson", alpha=0.12, lw=0)

ax_price.plot(bar_stamps, bars["close"].to_numpy(), color="#111827", lw=1.0)
ax_price.set_title("BTC tape: high-VPIN regimes shaded", fontsize=12, fontweight="bold")
ax_price.set_ylabel("Price (USD)")
ax_price.grid(True, alpha=0.25)

ax_vpin.plot(bar_stamps, vpin_values, color="#b91c1c", lw=1.1, label=f"VPIN({VPIN_PERIOD})")
ax_vpin.plot(
    bar_stamps,
    vpin_threshold_values,
    color="#6b7280",
    ls="--",
    lw=1.0,
    label=f"Causal {THRESHOLD_WINDOW}-bar 80th pct",
)
ax_vpin.set_ylabel("VPIN")
ax_vpin.set_xlabel("Time")
ax_vpin.set_ylim(0, 1)
ax_vpin.legend(loc="upper left", fontsize=9)
ax_vpin.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
q = quintiles["quintile"].to_numpy()
ax.bar(q, quintiles["mean_fwd_bps"].to_numpy(), color=["#9ca3af"] * 4 + ["#16a34a"], width=0.65)
ax.axhline(0, color="#374151", lw=0.8)
ax.set_xticks(q)
ax.set_xticklabels([f"Q{i + 1}" for i in q])
ax.set_xlabel("Normalized OFI quintile (Q5 = strongest buy imbalance)")
ax.set_ylabel("Mean next-60s return (bps)")
ax.set_title(
    f"Order flow imbalance leads short-horizon returns (corr = {corr_60s:+.3f})",
    fontsize=12,
    fontweight="bold",
)
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Takeaways

- **Sample by information, not by clock.** Volume bars put the informed bursts where the statistics can see them; on one-minute clock bars the same tape is a much weaker signal.
- **VPIN is a risk dial, not a directional signal.** It tells you when adverse selection is expensive - widen quotes, cut size, tighten stops - not which way to trade. Mean VPIN during bursts sits clearly above the quiet tape; that gap is the whole point.
- **OFI is directional and short-lived.** The edge decays over minutes; anything slower than the bar clock you compute it on is a different trade.
- **Synthetic data has a known ground truth.** Because we generated the informed flow, we can check that the estimator recovers it - something real tape never lets you do.

### Next steps

- Replace the synthetic tape with exchange trade+book snapshots (e.g. Binance `aggTrades` and `bookTicker`), keeping the same three function calls.
- Compare VPIN on volume bars vs dollar bars (`df.anta.resample_dollar_bars`) on the same tape.
- Feed `vpin` into `df.anta.backtest(...)` as a position-size multiplier instead of a standalone signal.
